# Flood Probability Estimation using AutoML — Balikpapan & Samarinda (Tabular Local)

Notebook ini memakai [processed/dataset_utama.csv](processed/dataset_utama.csv) sebagai dataset utama training. Layer RBI, DEM, dan batas administrasi tetap disiapkan sebagai sumber fitur spasial dan untuk peta probabilitas, tetapi alur model utama tidak lagi bergantung pada pipeline geospasial berat di notebook lama.

## 0. Setup environment
Pasang dependensi jika perlu lalu impor semua modul yang dipakai. Notebook ini dirancang tetap jalan di Windows dan menyesuaikan konflik PROJ/GDAL bila ada.

In [1]:
# Jalankan sekali bila environment belum siap.
# %pip install numpy pandas geopandas rasterio shapely scikit-learn xgboost catboost flaml matplotlib scipy pyproj fiona pyogrio rtree openpyxl

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from flaml import AutoML

import geopandas as gpd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import rasterize
from rasterio.mask import mask as rio_mask
from scipy.ndimage import distance_transform_edt
from shapely.geometry import Point, box

warnings.filterwarnings("ignore")

for _k in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(_k, None)

try:
    import pyproj
    os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
    pyproj.datadir.set_data_dir(pyproj.datadir.get_data_dir())
except Exception:
    pass

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Import OK. SHAP:", HAS_SHAP)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\noels\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\noels\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\noels\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _ARRAY_API not found

Import OK. SHAP: True


## 1. Configuration
Atur path proyek, dataset utama, dan parameter sampling. Dataset utama yang dipakai untuk training berada di `processed/dataset_utama.csv`.

In [2]:
BASE_DIR = Path.cwd()
CLEAN_DIR = BASE_DIR / "clean"
OUT_DIR = BASE_DIR / "outputs"
PROCESSED_DIR = BASE_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DATA_PATH = PROCESSED_DIR / "dataset_utama.csv"
RAW_BNPB_PATH = BASE_DIR / "dataset" / "bnpb-banjir_Smdbpp.csv"
RAW_WEATHER_PATH = BASE_DIR / "dataset" / "data-cuaca_smdbpp16-26.csv"
DEM_PATH = BASE_DIR / "dem" / "DEM SRTM 30M KALIMANTAN TIMUR.tif"
ADMIN_DIR = CLEAN_DIR / "admin"

RANDOM_SEED = 42
TIME_BUDGET = 120
UTM_CRS = "EPSG:32750"
WGS84 = "EPSG:4326"
USE_MAIN_DATASET = True
USE_RAW_GEOSPATIAL = True
np.random.seed(RANDOM_SEED)

CITIES = {
    "Balikpapan": dict(
        gdb=BASE_DIR / "RBI" / "KotaBalikpapan" / "RBI50K_KOTA BALIKPAPAN_KUGI50.gdb",
        kabkota="Kota Balikpapan",
        banjir=CLEAN_DIR / "banjir_balikpapan.csv",
        penduduk=CLEAN_DIR / "penduduk_balikpapan.csv",
        admin_kec=ADMIN_DIR / "balikpapan_kecamatan.geojson",
        admin_kel=ADMIN_DIR / "balikpapan_kelurahan.geojson",
    ),
    "Samarinda": dict(
        gdb=BASE_DIR / "RBI" / "KotaSamarinda" / "RBI50K_KOTA SAMARINDA_KUGI50.gdb",
        kabkota="Kota Samarinda",
        banjir=CLEAN_DIR / "banjir_samarinda.csv",
        penduduk=CLEAN_DIR / "penduduk_samarinda.csv",
        admin_kec=ADMIN_DIR / "samarinda_kecamatan.geojson",
        admin_kel=ADMIN_DIR / "samarinda_kelurahan.geojson",
    ),
}

ADMIN_KEC_NAMECOL = "WADMKC"
ADMIN_KEL_NAMECOL = "WADMKD"
L_DESAKEL = "ADMINISTRASI_AR_DESAKEL"
L_KECAMATAN = "ADMINISTRASI_AR_KECAMATAN"
L_KABKOTA = "ADMINISTRASI_AR_KABKOTA"
L_RIVER = "SUNGAI_LN_50K"
L_ROAD = "JALAN_LN_50K"
L_LANDCOVER = "PENUTUPLAHAN_AR_50K"
COL_KABKOTA = "WADMKK"
COL_KECAMATAN = "WADMKC"
COL_KELURAHAN = "WADMKD"
LC_CLASS_COL = "REMARK"

POINTS_PER_EVENT = 8
FLOOD_ELEV_QUANTILE = 0.40
NONFLOOD_ELEV_QUANTILE = 0.50
NONFLOOD_MIN_DIST = 750.0

NUMERIC_COLS = ["elevation", "slope", "dist_river", "dist_road", "pop_density"]
CAT_COLS = ["landcover", "city"]
FEATURE_COLS = NUMERIC_COLS + CAT_COLS
print("Konfigurasi dimuat.")
print("Dataset utama:", TRAIN_DATA_PATH)

Konfigurasi dimuat.
Dataset utama: c:\MyFolder\ProjekTipis\machine-learning\processed\processed\dataset_utama.csv


## 2. Geospatial helper functions
Bagian ini dipakai untuk membangun titik latih dari data mentah dan untuk membuat peta probabilitas per kota. Kalau kamu hanya ingin training dari dataset utama, sel-sel setelah ini tetap aman karena notebook akan memprioritaskan `dataset_utama.csv`.

In [3]:
def assert_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Tidak ditemukan: {path}")


def _norm(value):
    return str(value).strip().lower()


def read_gdb(gdb, layer):
    assert_file(gdb)
    gdf = gpd.read_file(gdb, layer=layer)
    if gdf.crs is None:
        gdf = gdf.set_crs(WGS84)
    return gdf.to_crs(UTM_CRS)


def load_admin(path, name_col):
    assert_file(path)
    gdf = gpd.read_file(path)
    if name_col not in gdf.columns:
        raise KeyError(f"Kolom {name_col!r} tidak ada di {path.name}. Kolom tersedia: {list(gdf.columns)}")
    if gdf.crs is None:
        gdf = gdf.set_crs(WGS84)
    gdf = gdf.to_crs(UTM_CRS).copy()
    gdf["adm_name"] = gdf[name_col]
    return gdf


def city_boundary_from_admin(admin_kec_gdf):
    return admin_kec_gdf.dissolve().geometry.iloc[0]


def clip_to_city(gdf, boundary_geom):
    return gdf[gdf.intersects(boundary_geom)].copy()


def clip_dem_to_bounds(dem_path, boundary_geom_wgs84, dst_crs=UTM_CRS, pad=0.02):
    minx, miny, maxx, maxy = boundary_geom_wgs84.bounds
    bb = box(minx - pad, miny - pad, maxx + pad, maxy + pad)
    with rasterio.open(dem_path) as src:
        arr, tr = rio_mask(src, [bb], crop=True)
        t2, w2, h2 = calculate_default_transform(
            src.crs, dst_crs, arr.shape[2], arr.shape[1],
            *rasterio.transform.array_bounds(arr.shape[1], arr.shape[2], tr)
        )
        dst = np.full((h2, w2), np.nan, dtype="float32")
        reproject(
            source=arr[0],
            destination=dst,
            src_transform=tr,
            src_crs=src.crs,
            dst_transform=t2,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
        )
    return dst.astype(float), t2


def compute_slope_deg(dem, transform):
    px, py = abs(transform.a), abs(transform.e)
    dzdy, dzdx = np.gradient(dem, py, px)
    return np.degrees(np.arctan(np.sqrt(dzdx ** 2 + dzdy ** 2)))


def sample_array_at_points(points_gdf, array, transform):
    values = []
    for geom in points_gdf.geometry:
        r, c = rowcol(transform, geom.x, geom.y)
        if 0 <= r < array.shape[0] and 0 <= c < array.shape[1]:
            values.append(array[r, c])
        else:
            values.append(np.nan)
    return np.array(values, dtype=float)


def attach_polygon_value(points_gdf, poly_gdf, value_col):
    poly = poly_gdf[["geometry", value_col]]
    joined = gpd.sjoin(points_gdf, poly, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]
    return joined.loc[points_gdf.index, value_col].values


def sample_flood_points_in_kecamatan(poly, n, dem, transform, elev_q, rng):
    h, w = dem.shape
    mask = rasterize([(poly, 1)], out_shape=(h, w), transform=transform, fill=0, all_touched=True).astype(bool)
    cells = mask & np.isfinite(dem)
    if cells.sum() == 0:
        return []
    threshold = np.nanquantile(dem[cells], elev_q)
    low_cells = cells & (dem <= threshold)
    rows, cols = np.where(low_cells if low_cells.sum() >= n else cells)
    if len(rows) == 0:
        return []
    idx = rng.choice(len(rows), size=min(n, len(rows)), replace=len(rows) < n)
    points = []
    for i in idx:
        x, y = rasterio.transform.xy(transform, rows[i], cols[i])
        points.append(Point(x, y))
    return points


def generate_nonflood_points(flood_gdf, dem, transform, n, elev_q, min_dist, rng):
    minx, miny, maxx, maxy = flood_gdf.total_bounds
    pad = 0.15 * max(maxx - minx, maxy - miny)
    minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad
    threshold = np.nanquantile(dem, elev_q)
    flood_union = flood_gdf.unary_union
    points, tries = [], 0
    while len(points) < n and tries < n * 400:
        tries += 1
        px, py = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        r, c = rowcol(transform, px, py)
        if not (0 <= r < dem.shape[0] and 0 <= c < dem.shape[1]):
            continue
        if not np.isfinite(dem[r, c]) or dem[r, c] < threshold:
            continue
        p = Point(px, py)
        if p.distance(flood_union) < min_dist:
            continue
        points.append(p)
    return gpd.GeoDataFrame(geometry=points, crs=UTM_CRS)

print("Helper geospasial siap.")

Helper geospasial siap.


## 3. Load main dataset and optional geospatial rebuild
Alur utama memakai `processed/dataset_utama.csv`. Kalau data mentah masih tersedia, notebook juga bisa membangun ulang titik berlabel per kota untuk kebutuhan peta probabilitas.

In [4]:
def load_main_training_data(path):
    assert_file(path)
    df = pd.read_csv(path)
    if "Flood" not in df.columns:
        raise KeyError("Kolom target 'Flood' tidak ditemukan di dataset utama.")
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df["year"] = df["time"].dt.year
        df["month"] = df["time"].dt.month
        df["dayofyear"] = df["time"].dt.dayofyear
        df["dayofweek"] = df["time"].dt.dayofweek
    if "city" not in df.columns:
        df["city"] = "unknown"
    if "location_id" not in df.columns:
        df["location_id"] = -1
    return df


df_main = load_main_training_data(TRAIN_DATA_PATH)
print("Dataset utama:", df_main.shape)
print(df_main.head(3).to_string())

RAW_AVAILABLE = USE_RAW_GEOSPATIAL and TRAIN_DATA_PATH.exists() and DEM_PATH.exists()
for cfg in CITIES.values():
    RAW_AVAILABLE = RAW_AVAILABLE and cfg["gdb"].exists() and cfg["admin_kec"].exists() and cfg["admin_kel"].exists() and cfg["banjir"].exists() and cfg["penduduk"].exists()

CITY_LAYERS = {}
RAW_POINT_DFS = []

if RAW_AVAILABLE:
    def build_city_dataset(city_name, cfg):
        print(f"\n========== {city_name} ==========")
        rng = np.random.default_rng(RANDOM_SEED)
        kec = load_admin(cfg["admin_kec"], ADMIN_KEC_NAMECOL)
        desakel = load_admin(cfg["admin_kel"], ADMIN_KEL_NAMECOL)
        boundary_utm = city_boundary_from_admin(kec)
        boundary_wgs = gpd.GeoSeries([boundary_utm], crs=UTM_CRS).to_crs(WGS84).iloc[0]
        dem, transform = clip_dem_to_bounds(DEM_PATH, boundary_wgs)
        slope = compute_slope_deg(dem, transform)
        rivers = clip_to_city(read_gdb(cfg["gdb"], L_RIVER), boundary_utm)
        roads = clip_to_city(read_gdb(cfg["gdb"], L_ROAD), boundary_utm)
        landcover = clip_to_city(read_gdb(cfg["gdb"], L_LANDCOVER), boundary_utm)

        pen = pd.read_csv(cfg["penduduk"])
        pen["_key"] = pen["kelurahan"].map(_norm)
        desakel["_key"] = desakel["adm_name"].map(_norm)
        desakel["area_km2"] = desakel.geometry.area / 1e6
        desakel = desakel.merge(pen[["_key", "value", "value_type"]].drop_duplicates("_key"), on="_key", how="left")
        desakel["pop_density"] = np.where(desakel["value_type"].eq("count"), desakel["value"] / desakel["area_km2"], desakel["value"])

        kec["_key"] = kec["adm_name"].map(_norm)
        banjir = pd.read_csv(cfg["banjir"])
        banjir["_key"] = banjir["kecamatan"].map(_norm)
        kec = kec.merge(banjir[["_key", "banjir_count"]], on="_key", how="left")

        flood_pts = []
        for _, row in kec.iterrows():
            cnt = row["banjir_count"]
            if pd.isna(cnt) or cnt <= 0:
                continue
            flood_pts.extend(sample_flood_points_in_kecamatan(row.geometry, int(round(cnt)) * POINTS_PER_EVENT, dem, transform, FLOOD_ELEV_QUANTILE, rng))
        flood_gdf = gpd.GeoDataFrame(geometry=flood_pts, crs=UTM_CRS)
        nonflood_gdf = generate_nonflood_points(flood_gdf, dem, transform, len(flood_gdf), NONFLOOD_ELEV_QUANTILE, NONFLOOD_MIN_DIST, rng)
        flood_gdf["label"] = 1
        nonflood_gdf["label"] = 0
        pts = gpd.GeoDataFrame(pd.concat([flood_gdf[["geometry", "label"]], nonflood_gdf[["geometry", "label"]]], ignore_index=True), geometry="geometry", crs=UTM_CRS)

        rivers_u, roads_u = rivers.unary_union, roads.unary_union
        pts["elevation"] = sample_array_at_points(pts, dem, transform)
        pts["slope"] = sample_array_at_points(pts, slope, transform)
        pts["dist_river"] = pts.geometry.apply(lambda g: g.distance(rivers_u))
        pts["dist_road"] = pts.geometry.apply(lambda g: g.distance(roads_u))
        pts["landcover_raw"] = attach_polygon_value(pts, landcover, LC_CLASS_COL).astype(str)
        pts["pop_density"] = attach_polygon_value(pts, desakel, "pop_density")
        pts["city"] = city_name
        out = pd.DataFrame(pts.drop(columns="geometry"))
        out["x"] = pts.geometry.x.values
        out["y"] = pts.geometry.y.values
        return out, dict(dem=dem, slope=slope, transform=transform, rivers=rivers, roads=roads, landcover=landcover, desakel=desakel, boundary=boundary_utm)

    for city_name, cfg in CITIES.items():
        city_df, layers = build_city_dataset(city_name, cfg)
        RAW_POINT_DFS.append(city_df)
        CITY_LAYERS[city_name] = layers
    df_points = pd.concat(RAW_POINT_DFS, ignore_index=True)
    print("Dataset raw titik:", df_points.shape)
else:
    df_points = pd.DataFrame()
    print("Data mentah geospasial tidak lengkap. Notebook akan fokus ke dataset utama untuk training.")

FileNotFoundError: Tidak ditemukan: c:\MyFolder\ProjekTipis\machine-learning\processed\processed\dataset_utama.csv

## 4. Combine datasets and encode categorical features
Bagian ini menyiapkan tabel model utama dari `dataset_utama.csv`. Kalau titik geospasial raw tersedia, notebook juga membangun encoding `landcover` untuk kebutuhan peta probabilitas.

In [ ]:
if RAW_AVAILABLE and not df_points.empty:
    df_model = df_points.copy()
    model_source = "raw_geospatial_points"
else:
    df_model = df_main.copy()
    model_source = "dataset_utama"

if "city" not in df_model.columns:
    df_model["city"] = "unknown"

if "time" in df_model.columns:
    df_model["time"] = pd.to_datetime(df_model["time"], errors="coerce")
    df_model["year"] = df_model["time"].dt.year
    df_model["month"] = df_model["time"].dt.month
    df_model["dayofyear"] = df_model["time"].dt.dayofyear
    df_model["dayofweek"] = df_model["time"].dt.dayofweek

if RAW_AVAILABLE and not df_points.empty:
    lc_classes = sorted(df_points["landcover_raw"].dropna().astype(str).unique().tolist())
    LC_MAP = {cls: idx for idx, cls in enumerate(lc_classes)}
    df_points["landcover"] = df_points["landcover_raw"].astype(str).map(LC_MAP).fillna(-1).astype(int)
else:
    LC_MAP = {}

print("Sumber training:", model_source)
print("df_model:", df_model.shape)
print("class balance:")
print(df_model["Flood"].value_counts().rename({0: "non_banjir", 1: "banjir"}).to_string())
print("ratio banjir/non_banjir:", round(df_model["Flood"].sum() / max(len(df_model) - df_model["Flood"].sum(), 1), 4))
display(df_model.head())

## 5. Exploratory analysis
Cek keseimbangan kelas, korelasi fitur numerik, dan ringkasan statistik singkat sebelum training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts = df_model["Flood"].value_counts().rename({0: "Non-Banjir", 1: "Banjir"})
class_counts.plot(kind="bar", ax=axes[0], color=["#4C9F70", "#D1495B"])
axes[0].set_title("Keseimbangan kelas")
axes[0].tick_params(axis="x", rotation=0)

numeric_candidates = [
    col for col in df_model.columns
    if col not in {"Flood", "time", "city"} and pd.api.types.is_numeric_dtype(df_model[col])
]
corr = df_model[numeric_candidates + ["Flood"]].corr(numeric_only=True)
im = axes[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr.columns)))
axes[1].set_xticklabels(corr.columns, rotation=45, ha="right")
axes[1].set_yticks(range(len(corr.columns)))
axes[1].set_yticklabels(corr.columns)
axes[1].set_title("Korelasi fitur numerik")
for (row, col), value in np.ndenumerate(corr.values):
    axes[1].text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

display(df_model[numeric_candidates + ["Flood"]].describe().round(3))

## 6. Preprocessing & feature engineering
Kolom yang dipakai untuk training diambil dari tabel utama. Fitur numerik di-scale, fitur kategori di-encode, lalu kita siapkan fungsi `transform_features(frame)` untuk semua model.

In [ ]:
df_train = df_model.replace([np.inf, -np.inf], np.nan).copy()
feature_numeric_candidates = [
    "latitude",
    "longitude",
    "year",
    "month",
    "dayofyear",
    "dayofweek",
    "elevation",
    "slope",
    "dist_river",
    "dist_road",
    "pop_density",
    "precipitation_sum (mm)",
    "temperature_2m_max (°C)",
    "temperature_2m_min (°C)",
    "wind_speed_10m_max (km/h)",
    "rain_sum (mm)",
    "soil_moisture_0_to_100cm_mean (m³/m³)",
]
feature_categorical_candidates = ["city", "location_id", "landcover_raw"]

NUMERIC_COLS_TRAIN = [col for col in feature_numeric_candidates if col in df_train.columns]
CAT_COLS_TRAIN = [col for col in feature_categorical_candidates if col in df_train.columns]
FEATURE_COLS_TRAIN = NUMERIC_COLS_TRAIN + CAT_COLS_TRAIN

before_rows = len(df_train)
df_train = df_train.dropna(subset=FEATURE_COLS_TRAIN + ["Flood"]).reset_index(drop=True)
print(f"Baris dibuang (NaN/invalid): {before_rows - len(df_train)} | tersisa: {len(df_train)}")

scaler = StandardScaler().fit(df_train[NUMERIC_COLS_TRAIN]) if NUMERIC_COLS_TRAIN else None
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(df_train[CAT_COLS_TRAIN]) if CAT_COLS_TRAIN else None


def transform_features(frame):
    parts = []
    if NUMERIC_COLS_TRAIN:
        numeric_part = pd.DataFrame(
            scaler.transform(frame[NUMERIC_COLS_TRAIN]),
            columns=NUMERIC_COLS_TRAIN,
            index=frame.index,
        )
        parts.append(numeric_part)
    if CAT_COLS_TRAIN:
        cat_part = pd.DataFrame(
            encoder.transform(frame[CAT_COLS_TRAIN]),
            columns=CAT_COLS_TRAIN,
            index=frame.index,
        )
        parts.append(cat_part)
    return pd.concat(parts, axis=1)


X_all = transform_features(df_train)
y_all = df_train["Flood"].astype(int)
print("Matriks fitur:", X_all.shape)
print("Fitur:", list(X_all.columns))

## 7. Train/test split
Karena targetnya prediksi waktu banjir, split dibuat berbasis waktu bila kolom `time` tersedia. Kalau tidak ada, notebook fallback ke split stratified.

In [ ]:
if "time" in df_train.columns and df_train["time"].notna().any():
    ordered = df_train.sort_values("time").reset_index(drop=True)
    cut_idx = int(len(ordered) * 0.70)
    train_df = ordered.iloc[:cut_idx].copy()
    test_df = ordered.iloc[cut_idx:].copy()
    split_mode = "time-aware"
else:
    train_df, test_df = train_test_split(
        df_train,
        test_size=0.30,
        stratify=df_train["Flood"],
        random_state=RANDOM_SEED,
    )
    split_mode = "stratified"

X_train_raw = train_df[FEATURE_COLS_TRAIN].copy()
X_test_raw = test_df[FEATURE_COLS_TRAIN].copy()
y_train = train_df["Flood"].astype(int).reset_index(drop=True)
y_test = test_df["Flood"].astype(int).reset_index(drop=True)

X_train_raw = X_train_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)
X_train = transform_features(X_train_raw)
X_test = transform_features(X_test_raw)

neg, pos = y_train.value_counts().get(0, 0), y_train.value_counts().get(1, 0)
scale_pos_weight = (neg / max(pos, 1)) if pos > 0 else 1.0
print("Split mode:", split_mode)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts().rename({0: "non_banjir", 1: "banjir"}).to_string())
print("Test class balance:")
print(y_test.value_counts().rename({0: "non_banjir", 1: "banjir"}).to_string())
print("scale_pos_weight:", round(scale_pos_weight, 3))

## 8. Baseline models
Random Forest, XGBoost, dan CatBoost dilatih dengan penanganan imbalance di data latih. Untuk model AutoML, notebook akan memakai subset latih yang lebih seimbang agar pencarian model tidak bias ke kelas mayoritas.

In [ ]:
def balance_training_subset(X, y, max_neg_pos_ratio=10, random_state=RANDOM_SEED):
    train_frame = X.copy()
    train_frame["__target__"] = y.values
    pos_df = train_frame[train_frame["__target__"] == 1]
    neg_df = train_frame[train_frame["__target__"] == 0]
    if len(pos_df) == 0 or len(neg_df) == 0:
        return X.copy(), y.copy()
    keep_neg = min(len(neg_df), max(len(pos_df) * max_neg_pos_ratio, len(pos_df)))
    neg_sample = neg_df.sample(n=keep_neg, random_state=random_state)
    balanced = pd.concat([pos_df, neg_sample], axis=0).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    y_bal = balanced.pop("__target__").astype(int)
    return balanced, y_bal


X_train_bal, y_train_bal = balance_training_subset(X_train, y_train, max_neg_pos_ratio=10)
print("Balanced train:", X_train_bal.shape)
print(y_train_bal.value_counts().rename({0: "non_banjir", 1: "banjir"}).to_string())

rf_class_weight = {0: 1.0, 1: scale_pos_weight}
models = {}
models["Random Forest"] = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight="balanced_subsample",
).fit(X_train_bal, y_train_bal)
models["XGBoost"] = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    n_jobs=-1,
).fit(X_train_bal, y_train_bal)
models["CatBoost"] = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.05,
    random_seed=RANDOM_SEED,
    verbose=0,
    class_weights=[1.0, scale_pos_weight],
).fit(X_train_bal, y_train_bal)
print("Terlatih:", list(models.keys()))

## 9. AutoML (FLAML)
FLAML dipakai sebagai pencari kandidat terbaik. Kalau CatBoost gagal di mesin tertentu, notebook otomatis retry tanpa CatBoost agar proses tetap jalan.

In [ ]:
automl = AutoML()
automl_settings = dict(
    task="classification",
    metric="roc_auc",
    estimator_list=["rf", "xgboost", "catboost"],
    time_budget=TIME_BUDGET,
    eval_method="cv",
    n_splits=5,
    seed=RANDOM_SEED,
    verbose=1,
)
try:
    automl.fit(X_train=X_train_bal, y_train=y_train_bal, **automl_settings)
except Exception as exc:
    print("Ulang tanpa catboost:", exc)
    automl_settings["estimator_list"] = ["rf", "xgboost"]
    automl.fit(X_train=X_train_bal, y_train=y_train_bal, **automl_settings)

print("Estimator terbaik:", automl.best_estimator)
print("Konfigurasi terbaik:", automl.best_config)
models["AutoML (FLAML)"] = automl

## 10. Evaluation
Gunakan AUC-ROC, AUC-PR, precision, recall, F1, balanced accuracy, ROC curve, dan confusion matrix. Untuk data banjir yang timpang, recall dan AUC-PR biasanya lebih informatif daripada accuracy.

In [ ]:
def get_proba(model, Xte):
    return np.asarray(model.predict_proba(Xte))[:, 1]


def evaluate_model(name, model, Xte, yte):
    proba = get_proba(model, Xte)
    pred = (proba >= 0.5).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(yte, pred),
        "Balanced_Accuracy": balanced_accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred, zero_division=0),
        "Recall": recall_score(yte, pred, zero_division=0),
        "F1": f1_score(yte, pred, zero_division=0),
        "AUC_ROC": roc_auc_score(yte, proba),
        "AUC_PR": average_precision_score(yte, proba),
    }, proba


results, probas = [], {}
for name, model in models.items():
    metrics, proba = evaluate_model(name, model, X_test, y_test)
    results.append(metrics)
    probas[name] = proba

res_df = pd.DataFrame(results).set_index("Model").sort_values("AUC_PR", ascending=False)
best_name = res_df.index[0]
best_model = models[best_name]
print("Model terbaik (AUC-PR tertinggi):", best_name)
display(res_df.round(4))

plt.figure(figsize=(7, 6))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Kurva ROC")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_pred = (get_proba(best_model, X_test) >= 0.5).astype(int)
cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Non-Banjir", "Banjir"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Non-Banjir", "Banjir"])
ax.set_xlabel("Prediksi")
ax.set_ylabel("Aktual")
ax.set_title(f"Confusion Matrix — {best_name}")
for (row, col), value in np.ndenumerate(cm):
    ax.text(col, row, str(value), ha="center", va="center", color="white" if value > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_pred, target_names=["Non-Banjir", "Banjir"]))

## 11. Feature importance
Ambil importance dari model yang mendukungnya, normalisasi, lalu agregasi supaya kamu bisa lihat fitur mana yang paling berpengaruh.

In [ ]:
importance_frames = []
for name, model in models.items():
    importance = None
    if hasattr(model, "feature_importances_"):
        importance = np.asarray(model.feature_importances_)
    elif name == "AutoML (FLAML)":
        try:
            estimator = getattr(getattr(model, "model", None), "estimator", None)
            if estimator is not None and hasattr(estimator, "feature_importances_"):
                importance = np.asarray(estimator.feature_importances_)
        except Exception:
            importance = None
    if importance is not None and len(importance) == X_train.shape[1]:
        importance_frames.append(pd.Series(importance / max(importance.sum(), 1e-12), index=X_train.columns, name=name))

if importance_frames:
    importance_df = pd.concat(importance_frames, axis=1)
    importance_df["mean"] = importance_df.mean(axis=1)
    importance_df = importance_df.sort_values("mean")
    importance_df["mean"].plot(kind="barh", figsize=(8, 4), color="#2A6F97")
    plt.title("Feature importance rata-rata")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    display(importance_df.round(4))
else:
    importance_df = pd.DataFrame()
    print("Tidak ada model yang menyediakan feature_importances_.")

## 12. Build probability maps per city
Peta probabilitas hanya dibuat kalau mode geospasial tersedia. Kalau notebook jatuh ke `dataset_utama.csv`, bagian ini akan dilewati dengan aman.

In [ ]:
def build_probability_map(city_name, layers):
    if not RAW_AVAILABLE or df_points.empty:
        print(f"Skip peta {city_name}: mode geospasial tidak aktif.")
        return

    dem = layers["dem"]
    slope = layers["slope"]
    transform = layers["transform"]
    rivers = layers["rivers"]
    roads = layers["roads"]
    landcover = layers["landcover"]
    desakel = layers["desakel"]
    h, w = dem.shape
    px = abs(transform.a)

    river_mask = rasterize(((geom, 1) for geom in rivers.geometry), out_shape=(h, w), transform=transform, fill=0, all_touched=True)
    road_mask = rasterize(((geom, 1) for geom in roads.geometry), out_shape=(h, w), transform=transform, fill=0, all_touched=True)
    dist_river_grid = distance_transform_edt(river_mask == 0) * px
    dist_road_grid = distance_transform_edt(road_mask == 0) * px

    lc_grid = rasterize(
        ((geom, LC_MAP.get(str(value), -1)) for geom, value in zip(landcover.geometry, landcover[LC_CLASS_COL])),
        out_shape=(h, w),
        transform=transform,
        fill=-1,
        dtype="float32",
    )
    pop_grid = rasterize(
        ((geom, float(value)) for geom, value in zip(desakel.geometry, desakel["pop_density"]) if pd.notna(value)),
        out_shape=(h, w),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )

    grid = pd.DataFrame({
        "elevation": dem.ravel(),
        "slope": slope.ravel(),
        "dist_river": dist_river_grid.ravel(),
        "dist_road": dist_road_grid.ravel(),
        "pop_density": pop_grid.ravel(),
        "landcover_raw": pd.Series(lc_grid.ravel()).map({v: k for k, v in LC_MAP.items()}).fillna("unknown").astype(str),
        "city": city_name,
    })
    valid_mask = grid[FEATURE_COLS_TRAIN].notna().all(axis=1).values
    proba = np.full(len(grid), np.nan, dtype=float)
    if valid_mask.sum() > 0:
        proba[valid_mask] = get_proba(best_model, transform_features(grid.loc[valid_mask, FEATURE_COLS_TRAIN + [c for c in ["landcover_raw", "city"] if c in grid.columns]]))
    prob_map = proba.reshape(h, w)

    tif_path = OUT_DIR / f"flood_probability_{city_name}.tif"
    png_path = OUT_DIR / f"flood_probability_{city_name}.png"
    with rasterio.open(
        tif_path,
        "w",
        driver="GTiff",
        height=h,
        width=w,
        count=1,
        dtype="float32",
        crs=UTM_CRS,
        transform=transform,
        nodata=np.nan,
    ) as dst:
        dst.write(prob_map.astype("float32"), 1)

    plt.figure(figsize=(8, 8))
    im = plt.imshow(prob_map, cmap="turbo", vmin=0, vmax=1)
    plt.colorbar(im, label="Probabilitas banjir")
    plt.title(f"Peta Probabilitas Banjir — {city_name} ({best_name})")
    plt.axis("off")
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Tersimpan:", tif_path, "&", png_path)


if RAW_AVAILABLE and not df_points.empty:
    for city_name, layers in CITY_LAYERS.items():
        build_probability_map(city_name, layers)
else:
    print("Peta probabilitas dilewati karena notebook berjalan di mode dataset utama.")

## 13. Save outputs
Simpan hasil evaluasi, tabel training yang dipakai, dan daftar file artefak yang tersedia di `outputs/`.

In [ ]:
res_df.to_csv(OUT_DIR / "model_comparison.csv")
df_train.to_csv(OUT_DIR / "training_table_used.csv", index=False)
if importance_df is not None and not importance_df.empty:
    importance_df.to_csv(OUT_DIR / "feature_importance.csv")
if RAW_AVAILABLE and not df_points.empty:
    df_points.to_csv(OUT_DIR / "labelled_points_geospatial.csv", index=False)
print("Output di", OUT_DIR)
print(sorted([p.name for p in OUT_DIR.iterdir()]))